## colesの緯度・経度を求めよう！！

[Testing REST APIs](https://blog.jetbrains.com/idea/2025/04/how-to-use-kotlin-notebooks-for-productive-development/#:~:text=Kotlin%20Notebook%20provides%20a%20powerful,available%20through%20the%20http%20variable.)

- Kotlin Notebook (KN) はRESTful APIも可能
- 始めるためには`%use ktor-client`が必要。下記が含まれる：
    - Ktor-based HTTP client
    - `kotlin.serialization`

In [2]:
%use dataframe
%use ktor-client

In [3]:
// const
object DirectoryPath {
    const val output = "./output/"
}

Overpass APIでオーストラリア全土のJSONを取得する

In [4]:
import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import kotlinx.coroutines.runBlocking

val http = HttpClient(CIO) {
    install(HttpTimeout) {
        requestTimeoutMillis = 300_000
        connectTimeoutMillis = 60_000
        socketTimeoutMillis = 300_000
    }
}

val areaId = "3600080500" // オーストラリア全土
val brand = "Coles"

fun buildOverpassQuery(areaId: String, brand: String): String = """
   [out:json][timeout:300];
   area(${areaId})->.searchArea;
   nwr["shop"="supermarket"]["brand"="$brand"](area.searchArea);
   out center qt;
""".trimIndent()

val result = runBlocking {
    val query = buildOverpassQuery(areaId, brand)

    val response = http.post("https://overpass-api.de/api/interpreter") {
        contentType(ContentType.Application.FormUrlEncoded)
        setBody("data=" + query)
    }
    response.bodyAsText()
}

Data Classのリストを作成し、必要な情報を格納する
必須なのは
- ID
- 緯度
- 経度

あると嬉しいのは
- 州名 (state)
- 郊外、地区 (suburb)
- 番地、通り (street)

In [5]:
import kotlinx.serialization.json.*

/**
 * tagsに属するstate, suburb, streetは、Overpass APIで取れるデータにはnullであることが
 * しばしばある。人間としては住所がわかりやすいが、ひとまず緯度・経度が分かれば十分なので、
 * state, suburb, streetをnullableにしている。
 */
data class ColesLocation(
    val id: Long,
    val lat: Double,            // 緯度
    val lon: Double,            // 経度
    val state: String? = null,  // 州
    val suburb: String? = null, // 郊外、地区、町レベル
    val street: String? = null  // 通り、番地レベル
)

val root = Json.parseToJsonElement(result).jsonObject
val elements = root["elements"]!!.jsonArray

val locations = elements.mapNotNull { element ->
    val obj = element.jsonObject

    val id = obj["id"]?.jsonPrimitive?.longOrNull
    val lat = obj["lat"]?.jsonPrimitive?.doubleOrNull
    val lon = obj["lon"]?.jsonPrimitive?.doubleOrNull

    if (id == null || lat == null || lon == null) {
        null
    } else {

        val tags = obj["tags"]?.jsonObject

        val state = tags?.get("addr:state")?.jsonPrimitive?.contentOrNull
        val suburb = tags?.get("addr:suburb")?.jsonPrimitive?.contentOrNull
        val street = tags?.get("addr:street")?.jsonPrimitive?.contentOrNull

        ColesLocation(
            id = id,
            lat = lat,
            lon = lon,
            state = state,
            suburb = suburb,
            street = street
        )
    }
}

Data Classに格納したデータをCSVに保存する

In [6]:
import java.io.File

fun exportToCsv(locations: List<ColesLocation>, filePath: String) {
    val file = File(filePath)

    file.bufferedWriter().use { writer ->
        // header
        writer.appendLine("id,lat,lon,state,suburb,street")

        // data row
        for (loc in locations) {
            val line = listOf(
                loc.id.toString(),
                loc.lat.toString(),
                loc.lon.toString(),
                loc.state ?: "",
                loc.suburb ?: "",
                loc.street ?: "",
            ).joinToString(",")

            writer.appendLine(line)
        }
    }
}

exportToCsv(locations, DirectoryPath.output + "coles_locations.csv")